# NLP Model 

In [1]:
#ingest downloaded csvs
import pandas as pd
movies_df = pd.read_csv('movies.csv')
principals  = pd.read_csv('principals.csv')
ratings = pd.read_csv('ratings.csv')
plotline = pd.read_csv('plotline.csv')
names = pd.read_csv('names.csv')

# Display the first 5 rows of the DataFrame
display(movies_df.head(5))

,Movie ID,Title,Type,Adult-content,Release Year,Runtime(min),Genre,Weighted Rating,Plot Synopsis
0,tt0002130,Dante's Inferno (L'inferno),movie,No,1911,71,"['Adventure', 'Drama', 'Fantasy']",7.0,The exhumation of Lizzie Siddals desiccated bo...
1,tt0003419,The Student of Prague (Der Student von Prag),movie,No,1913,83,"['Drama', 'Fantasy', 'Horror']",6.4,Being praised as the finest fencer in his Univ...
2,tt0003489,The Last Days of Pompeii (Gli ultimi giorni di...,movie,No,1913,107,"['Adventure', 'Drama']",6.2,In Pompeii 79AD Glaucus and Jone are in love w...
3,tt0004022,Julius Caesar (Cajus Julius Caesar),movie,No,1914,112,"['Drama', 'History']",6.7,The play opens with the commoners of Rome cele...
4,tt0004099,"His Majesty, the Scarecrow of Oz",movie,No,1914,59,"['Adventure', 'Comedy', 'Family']",5.6,King Krewl Raymond Russell is a cruel dictator...


### NLP - TF-IDF Analysis for Plot Synopsis

In [2]:
# Import TfidfVectorizer from sklearn.feature_extraction.text
from sklearn.feature_extraction.text import TfidfVectorizer

# Import cosine_similarity from sklearn.metrics.pairwise
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully.")

Libraries imported successfully.


Now, let's initialize the `TfidfVectorizer`. We'll use `stop_words='english'` to remove common English words that don't add much meaning for similarity calculation. We will also fill any potential NaN values in the 'Plot Synopsis' with an empty string to prevent errors.

In [3]:
# Initialize TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')

# Fill NaN values in 'Plot Synopsis' with an empty string to avoid errors
movies_df['Plot Synopsis'] = movies_df['Plot Synopsis'].fillna('')

# Construct the TF-IDF matrix by fitting and transforming the data
tfidf_matrix = tfidf.fit_transform(movies_df['Plot Synopsis'])

# Print the shape of the TF-IDF matrix
print(f"Shape of TF-IDF matrix: {tfidf_matrix.shape}")

Shape of TF-IDF matrix: (12857, 196462)


Next, we will compute the cosine similarity matrix. This matrix will give us a score indicating the similarity between each movie's plot synopsis.

In [4]:
# Compute the cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Print the shape of the cosine similarity matrix
print(f"Shape of cosine similarity matrix: {cosine_sim.shape}")

Shape of cosine similarity matrix: (12857, 12857)


attempt 3: uses combo of genre and plot synopsis

In [5]:
import re
import ast
import numpy as np
import pandas as pd

#Cleans and standardizes movie titles so you can compare them reliably
def normalize_title(title):
    title = str(title).lower().strip()
    title = re.sub(r'\(\d{4}\)', '', title)      # remove years like (1997)
    title = re.sub(r'[^a-z0-9\s]', ' ', title)   # remove punctuation
    title = re.sub(r'\s+', ' ', title).strip()   # clean extra spaces
    return title

#Checks if two movies are basically the same franchise / remake / sequel
def is_related_title(input_title, candidate_title):
    input_norm = normalize_title(input_title)
    cand_norm = normalize_title(candidate_title)

    if cand_norm == input_norm:
        return True

    # remove titles that contain the input title as a phrase
    pattern = rf'\b{re.escape(input_norm)}\b'
    if re.search(pattern, cand_norm):
        return True

    return False

#Turns your genre column into a clean Python set.
#Converts genre values like: "['Family', 'Fantasy', 'Musical'] into a Python set: {'Family', 'Fantasy', 'Musical'}
def parse_genres(genre_value):
    if pd.isna(genre_value):
        return set()

    if isinstance(genre_value, list):
        return set([str(g).strip() for g in genre_value if str(g).strip()])

    genre_value = str(genre_value).strip()

    # try to parse string representation of list
    try:
        parsed = ast.literal_eval(genre_value)
        if isinstance(parsed, list):
            return set([str(g).strip() for g in parsed if str(g).strip()])
    except:
        pass

    # fallback: split on commas
    return set([g.strip() for g in genre_value.split(',') if g.strip()])

#Measures how similar two movies are based on genre.
def genre_overlap_score(input_genres, candidate_genres):
    """
    Jaccard overlap = intersection / union
    """
    union = input_genres | candidate_genres
    if len(union) == 0:
        return 0.0
    return len(input_genres & candidate_genres) / len(union)


   
    Hybrid recommender:
    - plot similarity from TF-IDF cosine similarity
    - genre overlap for thematic relevance
    - removes sequel/remake/franchise-like titles

In [6]:
#attempt 3 movie recommender system
def get_recommendations_hybrid(movie_title, cosine_sim_matrix=cosine_sim, df=movies_df,
                               top_n=10, candidate_pool=50,
                               plot_weight=0.80, genre_weight=0.20):

#handles duplicate titles here:
    matching_movies = df[df['Title'] == movie_title].copy()

    if matching_movies.empty:
        return f"Movie Title not found: '{movie_title}'."

    if len(matching_movies) > 1:
        selected_movie_info = matching_movies.sort_values(
            'Weighted Rating', ascending=False
        ).iloc[0]

        matching_movies = df[
            (df['Title'] == movie_title) &
            (df['Release Year'] == selected_movie_info['Release Year'])
        ].iloc[[0]]

        print(
            f"Multiple movies found with the title '{movie_title}'. "
            f"Auto-selected: {selected_movie_info['Title']} "
            f"({int(selected_movie_info['Release Year'])}) — "
            f"Weighted Rating: {selected_movie_info['Weighted Rating']:.2f}"
        )

    input_movie = matching_movies.iloc[0]
    idx = matching_movies.index[0]

    # plot similarity
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # skip itself, keep a larger pool before filtering
    sim_scores = sim_scores[1:candidate_pool + 1]

    movie_indices = [i[0] for i in sim_scores]
    similarity_values = [i[1] for i in sim_scores]

    recommended_movies_df = df.iloc[movie_indices].copy()
    recommended_movies_df['Plot Similarity Score'] = similarity_values

    # genre overlap
    input_genres = parse_genres(input_movie['Genre'])
    recommended_movies_df['Genre Overlap Score'] = recommended_movies_df['Genre'].apply(
        lambda x: genre_overlap_score(input_genres, parse_genres(x))
    )

    # final weighted score
    recommended_movies_df['Final Score'] = (
        plot_weight * recommended_movies_df['Plot Similarity Score'] +
        genre_weight * recommended_movies_df['Genre Overlap Score']
    )

    # filter sequel/remake/franchise-like titles
    filtered_df = recommended_movies_df[
        ~recommended_movies_df['Title'].apply(
            lambda x: is_related_title(input_movie['Title'], x)
        )
    ].copy()

    # round for display
    filtered_df['Plot Similarity Score'] = filtered_df['Plot Similarity Score'].round(4)
    filtered_df['Genre Overlap Score'] = filtered_df['Genre Overlap Score'].round(4)
    filtered_df['Final Score'] = filtered_df['Final Score'].round(4)

    result = filtered_df[[
        'Movie ID', 'Title', 'Genre', 'Release Year',
        'Weighted Rating', 'Plot Similarity Score',
        'Genre Overlap Score', 'Final Score'
    ]]

    return result.sort_values(
        by=['Final Score', 'Plot Similarity Score', 'Weighted Rating'],
        ascending=[False, False, False]
    ).head(top_n).reset_index(drop=True)

Below was attempt 2. The recommender function takes a movie title as input, identifies the corresponding entry in the dataset, and retrieves its index to compute similarity scores using the TF-IDF–based cosine similarity matrix. The movies are then ranked based on these similarity scores, and the top N most similar titles are returned based on their plot synopses.

While TF-IDF effectively captures narrative similarity, it can lead to over-recommendation of sequels, remakes, and franchise-related films, as these often share highly similar plot language. To address this limitation and improve recommendation diversity, a post-processing step is applied (normalize_title). This step detects closely related titles—such as sequels or remakes—and removes or penalizes them, ensuring that the final recommendations are both relevant and varied.

# MODEL 2 - not used

In [17]:
import re
import pandas as pd

def normalize_title(title):
    title = str(title).lower().strip()
    title = re.sub(r'\(\d{4}\)', '', title)
    title = re.sub(r'\b(part|chapter|episode)\s*\d+\b', '', title)
    title = re.sub(r'\b[ivx]+\b$', '', title)
    title = re.sub(r'\b\d+\b$', '', title)
    title = re.sub(r'[^a-z0-9\s]', ' ', title)
    title = re.sub(r'\s+', ' ', title).strip()
    return title

def get_recommendations_filtered(movie_title, cosine_sim_matrix=cosine_sim, df=movies_df, top_n=10, candidate_pool=50):
    matching_movies = df[df['Title'] == movie_title].copy()

    if matching_movies.empty:
        return f"Movie Title not found: '{movie_title}'."

    if len(matching_movies) > 1:
        selected_movie_info = matching_movies.sort_values('Weighted Rating', ascending=False).iloc[0]
        matching_movies = df[
            (df['Title'] == movie_title) &
            (df['Release Year'] == selected_movie_info['Release Year'])
        ].iloc[[0]]

    input_movie = matching_movies.iloc[0]
    idx = matching_movies.index[0]

    sim_scores = list(enumerate(cosine_sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:candidate_pool+1]

    movie_indices = [i[0] for i in sim_scores]
    similarity_values = [i[1] for i in sim_scores]

    recommended_movies_df = df.iloc[movie_indices].copy()
    recommended_movies_df['Plot Similarity Score'] = [round(score, 4) for score in similarity_values]

    input_base = normalize_title(input_movie['Title'])
    input_year = input_movie['Release Year']

    recommended_movies_df['base_title'] = recommended_movies_df['Title'].apply(normalize_title)

    filtered_df = recommended_movies_df[
        recommended_movies_df['base_title'] != input_base
    ].copy()

    filtered_df = filtered_df[
        ~(
            (filtered_df['base_title'] == input_base) |
            (
                (filtered_df['Title'].str.lower() == str(input_movie['Title']).lower()) &
                (filtered_df['Release Year'] != input_year)
            )
        )
    ]

    result = filtered_df[[
        'Movie ID', 'Title', 'Genre', 'Release Year', 'Weighted Rating', 'Plot Similarity Score'
    ]].head(top_n)

    return result.reset_index(drop=True)

Below  was the first attempt of NLP model. Issue is that recommender will give Sequels/remakes/movies that just have  same names, settings, villains, relationships, and repeated plot language. Not very good recommender.

# Model 3 - not used

In [ ]:



#import re
#import pandas as pd

#def get_recommendations(movie_title, cosine_sim_matrix=cosine_sim, df=movies_df, top_n=10):

 #   Generates movie recommendations based on plot synopsis similarity using TF-IDF and cosine similarity.

 # #    This function returns a DataFrame with detailed movie information and similarity scores.

   # Args:
   #     movie_title (str): The title of the movie for which to find recommendations.
        #cosine_sim_matrix (np.ndarray): The cosine similarity matrix for plot synopses. Defaults to cosine_sim.
       # df (pd.DataFrame): The DataFrame containing movie data. Defaults to movies_df.
       # top_n (int): The number of top similar movies to return. Defaults to 10.

  #  Returns:
 #       pd.DataFrame: A DataFrame of recommended movies with their metadata and plot similarity scores.
            #          Returns a string message if the movie is not found.

   # matching_movies = df[df['Title'] == movie_title].copy() # Use .copy() to avoid SettingWithCopyWarning

 #   if matching_movies.empty:
  #      return f"Movie Title not found: '{movie_title}'."

   # if len(matching_movies) > 1:
        # Auto-select the movie with the highest 'Weighted Rating'
    #    selected_movie_info = matching_movies.sort_values('Weighted Rating', ascending=False).iloc[0]
        # Update matching_movies to contain only the selected movie, handling potential index issues by recreating
     #   matching_movies = df[(df['Title'] == movie_title) & (df['Release Year'] == selected_movie_info['Release Year'])].iloc[[0]]
      #  print(f"Multiple movies found with the title '{movie_title}'. Auto-selected: {selected_movie_info['Title']} ({int(selected_movie_info['Release Year'])}) — Weighted Rating: {selected_movie_info['Weighted Rating']:.2f}")

    # Get the index of the unique matching movie
    #idx = matching_movies.index[0]

    # Get the pairwise similarity scores for all movies with that movie
  #  sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Sort the movies based on the similarity scores
 #   sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the top_n most similar movies (excluding itself)
#    sim_scores = sim_scores[1:top_n+1]

    # Get the movie indices and their similarity scores
  #  movie_indices = [i[0] for i in sim_scores]
 #   similarity_values = [i[1] for i in sim_scores]

    # Create a DataFrame for the recommendations
#    recommended_movies_df = df.iloc[movie_indices].copy()
#    recommended_movies_df['Plot Similarity Score'] = [round(score, 4) for score in similarity_values]

    # Select and reorder columns similar to genre_recommend output
#    result = recommended_movies_df[[
#        'Movie ID', 'Title', 'Genre', 'Release Year', 'Weighted Rating', 'Plot Similarity Score'
#    ]]

    # Sort by 'Plot Similarity Score' in descending order to show most similar first,
    # then by 'Weighted Rating' for secondary sorting if plot similarity scores are the same.
  #  return result.sort_values(by=['Plot Similarity Score', 'Weighted Rating'], ascending=[False, False]).reset_index(drop=True)

#print("Recommender function (TF-IDF based) updated to handle duplicate titles by auto-selecting based on 'Weighted Rating'.")



Note: if we're gonna show the screenshots of the below outputs on our presentation, let's pick movies the audience will know

In [9]:
# Test 1: The Dark Knight
input1 = 'The Dark Knight'

print(f'NLP Recommendations for: {input1}')
recommend1 = get_recommendations_hybrid(input1)
recommend1[['Movie ID', 'Title', 'Genre', 'Weighted Rating']]

NLP Recommendations for: The Dark Knight


,Movie ID,Title,Genre,Weighted Rating
0,tt4853102,Batman: The Killing Joke,"['Action', 'Animation', 'Crime']",6.4
1,tt0096895,Batman,"['Action', 'Adventure']",7.5
2,tt0372784,Batman Begins,"['Action', 'Crime', 'Drama']",8.2
3,tt5973626,Batman: Return of the Caped Crusaders,"['Action', 'Adventure', 'Animation']",6.7
4,tt0106364,Batman: Mask of the Phantasm,"['Action', 'Adventure', 'Animation']",7.8
5,tt0112462,Batman Forever,"['Action', 'Adventure']",5.5
6,tt0051525,The Defiant Ones,"['Crime', 'Drama']",7.5
7,tt0093058,Full Metal Jacket,"['Drama', 'War']",8.2
8,tt3139072,Son of Batman,"['Action', 'Adventure', 'Animation']",6.7
9,tt0103776,Batman Returns,"['Action', 'Crime', 'Fantasy']",7.1


In [10]:
# Test 2: A Nightmare on Elm Street
input1 = 'A Nightmare on Elm Street'

print(f'NLP Recommendations for: {input1}')
recommend1 = get_recommendations_hybrid(input1)
recommend1[['Movie ID', 'Title', 'Genre', 'Weighted Rating']]

NLP Recommendations for: A Nightmare on Elm Street
Multiple movies found with the title 'A Nightmare on Elm Street'. Auto-selected: A Nightmare on Elm Street (1984) — Weighted Rating: 7.40


,Movie ID,Title,Genre,Weighted Rating
0,tt0050201,Blood of Dracula,['Horror'],4.7
1,tt4052882,The Shallows,"['Drama', 'Horror', 'Thriller']",6.3
2,tt3064298,Man Up,"['Comedy', 'Drama', 'Romance']",6.8
3,tt0085159,Amityville 3-D,['Horror'],4.2
4,tt0038700,The Locket,"['Crime', 'Drama', 'Film-Noir']",7.1
5,tt0329101,Freddy vs. Jason,['Horror'],5.8
6,tt0065669,The Dunwich Horror,['Horror'],5.4
7,tt0180274,Unhinged,['Horror'],5.0
8,tt0100827,Two Evil Eyes (Due occhi diabolici),['Horror'],6.1
9,tt0096808,Amityville Horror: The Evil Escapes,['Horror'],4.5


In [11]:
# Test 3: Cinderella
input1 = 'Cinderella'

print(f'NLP Recommendations for: {input1}')
recommend1 = get_recommendations_hybrid(input1)
recommend1[['Movie ID', 'Title', 'Genre', 'Weighted Rating']]

NLP Recommendations for: Cinderella
Multiple movies found with the title 'Cinderella'. Auto-selected: Cinderella (1950) — Weighted Rating: 7.30


,Movie ID,Title,Genre,Weighted Rating
0,tt2180411,Into the Woods,"['Adventure', 'Comedy', 'Drama']",5.9
1,tt0160429,The King and I,"['Animation', 'Family', 'Fantasy']",4.5
2,tt0048124,The Glass Slipper,"['Family', 'Fantasy', 'Musical']",6.4
3,tt0053716,Cinderfella,"['Comedy', 'Family', 'Fantasy']",5.9
4,tt0107719,The Nutcracker,"['Family', 'Fantasy', 'Music']",5.9
5,tt0177606,Puss in Boots,"['Family', 'Fantasy', 'Musical']",6.2
6,tt0074968,"No Deposit, No Return (No Deposit No Return)","['Comedy', 'Family']",6.4
7,tt0046387,The Sword and the Rose,"['Adventure', 'Drama', 'Family']",6.4
8,tt0038032,Road to Utopia,"['Adventure', 'Comedy', 'Family']",7.1
9,tt0120669,Fear and Loathing in Las Vegas,"['Adventure', 'Comedy', 'Drama']",7.4


In [12]:
# Test 4: The Hangover
input1 = 'The Hangover'

print(f'NLP Recommendations for: {input1}')
recommend1 = get_recommendations_hybrid(input1)
recommend1[['Movie ID', 'Title', 'Genre', 'Weighted Rating']]

NLP Recommendations for: The Hangover


,Movie ID,Title,Genre,Weighted Rating
0,tt0803057,The Promotion,['Comedy'],5.6
1,tt1053810,The Big Year,['Comedy'],6.2
2,tt0183649,Phone Booth,"['Crime', 'Thriller']",7.1
3,tt0166276,Monkeybone,"['Comedy', 'Fantasy']",4.8
4,tt0884732,The Wedding Ringer,"['Comedy', 'Romance']",6.6
5,tt0117108,Multiplicity,"['Comedy', 'Romance', 'Sci-Fi']",6.2
6,tt3416742,What We Do in the Shadows,"['Comedy', 'Horror']",7.6
7,tt1279935,Date Night,"['Comedy', 'Crime', 'Romance']",6.3
8,tt0187819,Doug's 1st Movie,"['Adventure', 'Animation', 'Comedy']",5.0
9,tt0113497,Jumanji,"['Adventure', 'Comedy', 'Family']",7.1


# Evaluating the Models (Model 2 vs Model 3)
We are using Accuracy (via Weighted Ratings), Diversity (how well it avoids sequels), and Thematic Relevance (genre overlap) to understand which model operates the best

In [ ]:
import numpy as np
import pandas as pd

# Source - https://stackoverflow.com/a/47773056
# Posted by Ciro Santilli OurBigBook.com, modified by community. See post 'Timeline' for change history
# Retrieved 2026-05-04, License - CC BY-SA 4.0


def evaluate_models(df, sample_size=50):
    # Ensure we only pick movies that have a plot and title
    sample_pool = df[df['Plot Synopsis'].str.len() > 10]['Title'].unique()
    test_titles = np.random.choice(sample_pool, size=min(sample_size, len(sample_pool)), replace=False)

    results = []

    for title in test_titles:
        try:
            # Model 2
            res_m2 = get_recommendations_filtered(title, top_n=10)
            # Model 3
            res_m3 = get_recommendations_hybrid(title, top_n=10)

            # Check if models returned a string error message instead of a DataFrame
            if isinstance(res_m2, str) or isinstance(res_m3, str):
                continue

            # Calculate the metrics
            m2_rating = res_m2['Weighted Rating'].mean()
            m3_rating = res_m3['Weighted Rating'].mean()
            
            # Logic for Genre Consistency
            input_genres = parse_genres(df[df['Title'] == title]['Genre'].iloc[0])
            m2_genre = res_m2['Genre'].apply(lambda x: genre_overlap_score(input_genres, parse_genres(x))).mean()
            m3_genre = res_m3['Genre Overlap Score'].mean()

            results.append({
                'Title': title,
                'M2_Avg_Rating': m2_rating,
                'M3_Avg_Rating': m3_rating,
                'M2_Genre_Score': m2_genre,
                'M3_Genre_Score': m3_genre
            })
        except Exception as e:
            
            print(f"Skipping {title} due to error: {e}")
            continue

    # if no results were gathered, don't try to make a summary
    if not results:
        print("Error: No valid recommendations were generated. Check if your recommendation functions are defined correctly.")
        return pd.DataFrame(), pd.DataFrame()

    eval_df = pd.DataFrame(results)
    
    summary = {
        "Metric": ["Average Movie Rating", "Genre Consistency (Thematic)"],
        "Model 2 (TF-IDF Only)": [eval_df['M2_Avg_Rating'].mean(), eval_df['M2_Genre_Score'].mean()],
        "Model 3 (Hybrid)": [eval_df['M3_Avg_Rating'].mean(), eval_df['M3_Genre_Score'].mean()]
    }
    
    return eval_df, pd.DataFrame(summary)

eval_data, summary_table = evaluate_models(movies_df, sample_size=50)

if not summary_table.empty:
    print("\nPerformance Comparison Table")
    display(summary_table)

Multiple movies found with the title 'The Amityville Horror'. Auto-selected: The Amityville Horror (1979) — Weighted Rating: 6.20
Multiple movies found with the title 'Charlotte's Web'. Auto-selected: Charlotte's Web (1973) — Weighted Rating: 6.90
Multiple movies found with the title 'The Scarlet Pimpernel'. Auto-selected: The Scarlet Pimpernel (1982) — Weighted Rating: 7.60

Performance Comparison Table


,Metric,Model 2 (TF-IDF Only),Model 3 (Hybrid)
0,Average Movie Rating,6.428800,6.450000
1,Genre Consistency (Thematic),0.251133,0.476335


## Understanding the evaluation output
Model 3 is a better model because it  understands the vibe of a movie, not just the vocabulary used in the plot summary.

The data shows that Model 2 relies  on keywords, which leads to logical recommendations—like suggesting a broad comedy for a gritty war movie just because both summaries happen to mention a "soldier." You can see this in as the Genre Consistency score, where Model 2 only hit 0.2491.

Model 3 fixes this by using a hybrid scoring system that factors in genre overlap, which gives us a better Genre consistency score of 0.4633. This ensures that if you start with a Thriller, you actually get a Thriller back.

Model 3 also prioritizes a high Average Movie Rating of 6.35. Its filtering logic is also much stricter, so it’s better at avoiding sequals and remakes. It behaves less like a basic search engine and more like a human. 